# Лабораторная работа №4 (№8 по практикуму)
## «Методы кластерного анализа»

**Выполнил:** студент гр. ИТП-41 Стасев Б.В.
**Принял:** ассистент Ястребов А.А.

**Цель работы:** изучить методы и алгоритмы разделения объектов на кластеры.

**Задание:**
1. Изучить основные особенности задач кластерного анализа.
2. Создать примеры объектов (не менее 10), а также значения признаков объекта (не менее трёх), подлежащего распознаванию.
3. Разработать программу, реализующую алгоритм кластеризации методом *K* средних.
4. Разработать программу, реализующую алгоритм кластеризации методом максимина.
5. Оценить вычислительную сложность алгоритмов.

> Запуск в Google Colab: «Среда выполнения» → «Выполнить все» (Ctrl+F9). Дополнительные библиотеки не нужны.

In [ ]:
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

### 1. Исходные данные

В качестве объектов выбраны **15 легковых автомобилей**. Метки классов заранее не задаются (обучение без учителя) — алгоритмы должны сами сгруппировать автомобили по сходству характеристик.

Каждый объект описывается тремя признаками:
* $X_1$ — мощность двигателя, л.с.;
* $X_2$ — расход топлива, л/100 км;
* $X_3$ — снаряжённая масса, кг.

Признаки имеют разную размерность и порядок величин, поэтому перед кластеризацией они приводятся к безразмерному виду делением на максимальное значение соответствующего признака.

In [ ]:
FEATURES = ['Мощность, л.с.', 'Расход, л/100 км', 'Масса, кг']

cars = [
    # название               мощность  расход  масса
    ('Kia Picanto',              84,     5.4,   1000),
    ('Hyundai i10',              67,     5.0,    950),
    ('Renault Twingo',           75,     4.9,    980),
    ('Fiat 500',                 69,     5.1,    940),
    ('Chevrolet Spark',          75,     5.5,   1010),
    ('Toyota Camry',            181,     7.8,   1550),
    ('Volkswagen Passat',       150,     6.5,   1500),
    ('Skoda Superb',            190,     7.2,   1560),
    ('Mazda 6',                 165,     6.8,   1450),
    ('Hyundai Sonata',          180,     7.9,   1520),
    ('Toyota Land Cruiser',     309,    12.5,   2600),
    ('Mitsubishi Pajero',       178,    10.8,   2200),
    ('Nissan Patrol',           400,    14.5,   2750),
    ('Jeep Grand Cherokee',     286,    11.4,   2200),
    ('Chevrolet Tahoe',         355,    13.4,   2600),
]

names = [c[0] for c in cars]
X_raw = np.array([c[1:] for c in cars], dtype=float)
N, M = X_raw.shape

df = pd.DataFrame(X_raw, columns=FEATURES, index=range(1, N + 1))
df.insert(0, 'Автомобиль', names)
df.index.name = '№'
print(f'Количество объектов N = {N}, количество признаков M = {M}')
display(df)

In [ ]:
# Приведение признаков к безразмерному виду (деление на максимум признака)
X_MAX = X_raw.max(axis=0)
X = X_raw / X_MAX

df_norm = pd.DataFrame(X.round(3), columns=['X1 (мощность)', 'X2 (расход)', 'X3 (масса)'],
                       index=range(1, N + 1))
df_norm.insert(0, 'Автомобиль', names)
df_norm.index.name = '№'
print('Максимальные значения признаков:', dict(zip(FEATURES, X_MAX)))
display(df_norm)

### 2. Общие функции

Расстояние между объектами — евклидово:
$$D(X_i, X_j) = \sqrt{\sum_{m=1}^{M} (X_{mi} - X_{mj})^2}.$$

In [ ]:
def dist(a, b):
    """Евклидово расстояние между двумя объектами."""
    return math.sqrt(sum((a[m] - b[m]) ** 2 for m in range(len(a))))


def assign(X, prototypes):
    """Отнесение каждого объекта к кластеру с ближайшим прототипом."""
    labels = []
    for x in X:
        d = [dist(x, p) for p in prototypes]
        labels.append(int(np.argmin(d)))
    return np.array(labels)


def members_str(labels, k):
    return ', '.join(str(i + 1) for i in np.where(labels == k)[0])


def proto_str(p):
    """Прототип в исходных единицах измерения."""
    r = p * X_MAX
    return f'({r[0]:.1f} л.с.; {r[1]:.2f} л/100км; {r[2]:.0f} кг)'


def cluster_table(labels, prototypes):
    """Итоговая таблица: состав кластеров и их прототипы."""
    rows = []
    for k in range(len(prototypes)):
        idx = np.where(labels == k)[0]
        p = prototypes[k] * X_MAX
        rows.append({
            'Кластер': k + 1,
            'Кол-во': len(idx),
            'Объекты': ', '.join(f'{i + 1}. {names[i]}' for i in idx),
            'Мощность': round(p[0], 1),
            'Расход': round(p[1], 2),
            'Масса': round(p[2]),
        })
    return pd.DataFrame(rows).set_index('Кластер')


COLORS = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red',
          'tab:purple', 'tab:brown', 'tab:pink', 'tab:olive']


def plot_clusters(X, labels, prototypes, title, new_objects=None, new_names=None):
    """3D-диаграмма кластеров и проекция на плоскость «мощность – масса»."""
    fig = plt.figure(figsize=(14, 5.8))
    ax = fig.add_subplot(1, 2, 1, projection='3d')
    ax2 = fig.add_subplot(1, 2, 2)
    for k in range(len(prototypes)):
        c = COLORS[k % len(COLORS)]
        pts = X[labels == k]
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=45, color=c, label=f'Кластер {k + 1}')
        ax.scatter(*prototypes[k], s=230, marker='X', color=c, edgecolor='black')
        ax2.scatter(pts[:, 0], pts[:, 2], s=60, color=c, label=f'Кластер {k + 1}')
        ax2.scatter(prototypes[k][0], prototypes[k][2], s=260, marker='X', color=c, edgecolor='black')
    for i, x in enumerate(X):
        ax.text(x[0], x[1], x[2], f' {i + 1}', fontsize=8)
        ax2.annotate(str(i + 1), (x[0], x[2]), textcoords='offset points', xytext=(5, 4), fontsize=8)
    if new_objects is not None:
        ax.scatter(new_objects[:, 0], new_objects[:, 1], new_objects[:, 2],
                   s=200, marker='*', color='black', label='Распознаваемые')
        ax2.scatter(new_objects[:, 0], new_objects[:, 2], s=260, marker='*', color='black',
                    label='Распознаваемые')
        for x, nm in zip(new_objects, new_names):
            ax2.annotate(nm, (x[0], x[2]), textcoords='offset points', xytext=(6, -12), fontsize=8)
    ax.set_xlabel('X1 мощность')
    ax.set_ylabel('X2 расход')
    ax.set_zlabel('X3 масса')
    ax.legend(fontsize=8, loc='upper left')
    ax2.set_xlabel('X1 — мощность (норм.)')
    ax2.set_ylabel('X3 — масса (норм.)')
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    ax2.set_title('Проекция на плоскость X1–X3 (X — прототипы)')
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

### 3. Метод *K* средних

Пошаговый алгоритм:
1. Номер итерации $s = 0$.
2. Задаётся количество кластеров $K$, для каждого кластера выбирается первоначальный прототип $P_k^0$.
3. $s = s + 1$.
4. Каждый объект относится к кластеру с ближайшим прототипом.
5. В каждом кластере вычисляется новый прототип $P_k^s$ — среднее арифметическое признаков объектов кластера.
6. Если $P_k^s = P_k^{s-1}$ для всех $k$ — конец, иначе переход к шагу 3.

In [ ]:
def kmeans(X, K, init_idx=None, verbose=True, max_iter=100):
    """Кластеризация методом K средних.
    init_idx — номера (с нуля) объектов, выбранных первоначальными прототипами;
    по умолчанию берутся первые K объектов."""
    # Шаги 1-2: s = 0, первоначальные прототипы
    if init_idx is None:
        init_idx = list(range(K))
    prototypes = [X[i].copy() for i in init_idx]
    s = 0
    if verbose:
        print(f'K = {K}. Первоначальные прототипы — объекты №', [i + 1 for i in init_idx])
    while True:
        s += 1                                          # шаг 3
        labels = assign(X, prototypes)                  # шаг 4
        new_prototypes = []                             # шаг 5
        for k in range(K):
            cluster = X[labels == k]
            new_prototypes.append(cluster.mean(axis=0) if len(cluster) else prototypes[k])
        if verbose:
            print(f'\nИтерация s = {s}')
            for k in range(K):
                print(f'  Кластер {k + 1}: объекты [{members_str(labels, k)}]; '
                      f'новый прототип {proto_str(new_prototypes[k])}')
        # Шаг 6: проверка совпадения прототипов на двух соседних итерациях
        same = all(np.allclose(p, q) for p, q in zip(prototypes, new_prototypes))
        prototypes = new_prototypes
        if same or s >= max_iter:
            break
    if verbose:
        print(f'\nПрототипы не изменились — алгоритм завершён за {s} итераций.')
    return labels, np.array(prototypes), s

По смыслу предметной области задаём $K = 3$ кластера: «городские малолитражки», «седаны среднего класса», «внедорожники». В качестве первоначальных прототипов берём по одному характерному представителю каждой группы: №1 Kia Picanto, №6 Toyota Camry, №11 Toyota Land Cruiser.

In [ ]:
K = 3
INIT = [0, 5, 10]          # объекты №1, №6, №11

km_labels, km_prototypes, km_iters = kmeans(X, K, INIT)
display(cluster_table(km_labels, km_prototypes))

In [ ]:
plot_clusters(X, km_labels, km_prototypes, f'Метод K средних (K = {K}, итераций: {km_iters})')

**Влияние выбора начальных прототипов.** Если взять первые *K* объектов выборки (все три — малолитражки), алгоритм сходится к тому же разбиению (меняется только нумерация кластеров), но за большее число итераций. Поэтому начальные прототипы лучше выбирать осмысленно.

In [ ]:
lbl2, pr2, it2 = kmeans(X, K, init_idx=None, verbose=False)
print(f'Начальные прототипы — объекты №1, №2, №3. Итераций: {it2}')
display(cluster_table(lbl2, pr2))

**Выбор числа кластеров.** Для разных $K$ вычислим суммарный квадрат расстояний от объектов до прототипов своих кластеров (внутрикластерный разброс $J$). Резкий излом графика («локоть») указывает на естественное число кластеров.

In [ ]:
def inertia(X, labels, prototypes):
    return sum(dist(X[i], prototypes[labels[i]]) ** 2 for i in range(len(X)))

ks = range(1, 7)
J = []
for k in ks:
    l, p, _ = kmeans(X, k, init_idx=None, verbose=False)
    J.append(inertia(X, l, p))
print(pd.DataFrame({'K': list(ks), 'J': np.round(J, 4)}).to_string(index=False))

plt.figure(figsize=(6, 3.6))
plt.plot(list(ks), J, 'o-')
plt.xlabel('Количество кластеров K')
plt.ylabel('Внутрикластерный разброс J')
plt.title('Метод «локтя»')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4. Метод максимина

Количество кластеров заранее не задаётся и определяется автоматически:
1. Первый объект становится прототипом первого кластера: $P_1 = X_1$, $K = 1$.
2. Наиболее удалённый от $P_1$ объект становится прототипом второго кластера, $K = 2$.
3. Пороговое расстояние $T = D(P_1, P_2) / 2$.
4. Каждый объект относится к кластеру с ближайшим прототипом.
5. В каждом кластере находится объект $Y_k$, наиболее удалённый от своего прототипа.
6. Если $D(P_k, Y_k) < T$ для всех кластеров — конец. Иначе каждый $Y_k$, нарушивший условие, становится прототипом нового кластера.
7. Новый порог $T = \dfrac{\sum_{i=1}^{K-1}\sum_{j=i+1}^{K} D(P_i, P_j)}{K(K-1)}$ (половина среднего расстояния между прототипами).
8. Возврат к шагу 4.

In [ ]:
def maximin(X, first=0, verbose=True):
    """Кластеризация методом максимина. Прототипы — сами объекты выборки."""
    n = len(X)
    proto = [first]                                              # шаг 1
    d1 = [dist(X[first], X[j]) for j in range(n)]                # шаг 2
    proto.append(int(np.argmax(d1)))
    T = dist(X[proto[0]], X[proto[1]]) / 2                       # шаг 3
    if verbose:
        print(f'P1 = объект №{proto[0] + 1}, наиболее удалённый от него P2 = объект №{proto[1] + 1}')
        print(f'T = D(P1, P2) / 2 = {T:.4f}')
    step = 0
    while True:
        step += 1
        P = [X[i] for i in proto]
        labels = assign(X, P)                                    # шаг 4
        new = []
        if verbose:
            print(f'\nШаг распределения {step}: K = {len(proto)}, T = {T:.4f}')
        for k in range(len(proto)):                              # шаг 5
            idx = np.where(labels == k)[0]
            far = int(max(idx, key=lambda j: dist(P[k], X[j])))
            d = dist(P[k], X[far])
            ok = d < T                                           # шаг 6
            if verbose:
                print(f'  Кластер {k + 1} (прототип №{proto[k] + 1}): объекты [{members_str(labels, k)}]; '
                      f'Y{k + 1} = №{far + 1}, D = {d:.4f} {"< T" if ok else ">= T -> новый кластер"}')
            if not ok:
                new.append(far)
        if not new:
            break
        proto += new
        K = len(proto)                                           # шаг 7
        T = sum(dist(X[proto[i]], X[proto[j]])
                for i in range(K - 1) for j in range(i + 1, K)) / (K * (K - 1))
    if verbose:
        print(f'\nУсловие D(Pk, Yk) < T выполнено для всех кластеров. Итог: K = {len(proto)}')
    return labels, np.array([X[i] for i in proto]), proto, T

In [ ]:
mm_labels, mm_prototypes, mm_proto_idx, mm_T = maximin(X)
print('Прототипы — объекты №', [i + 1 for i in mm_proto_idx])
display(cluster_table(mm_labels, mm_prototypes))

In [ ]:
plot_clusters(X, mm_labels, mm_prototypes, f'Метод максимина (K = {len(mm_prototypes)}, T = {mm_T:.3f})')

Прототипы метода максимина — это сами объекты выборки, причём «крайние» (№1 — самая лёгкая малолитражка, №13 — самый тяжёлый внедорожник). Поэтому границы кластеров смещаются: менее мощные седаны (№7, №9) оказываются ближе к малолитражке №1, а седаны №6, №8, №10 и средние внедорожники №12, №14 — к прототипу №12.

Максимин часто используют для определения **числа кластеров и начальных прототипов**, после чего разбиение уточняют методом *K* средних:

In [ ]:
ref_labels, ref_prototypes, ref_iters = kmeans(X, len(mm_proto_idx), init_idx=mm_proto_idx, verbose=False)
print(f'K средних с начальными прототипами из максимина: K = {len(mm_proto_idx)}, итераций: {ref_iters}')
display(cluster_table(ref_labels, ref_prototypes))

### 5. Распознавание новых объектов

Полученные прототипы кластеров используются для отнесения новых объектов: объект относится к кластеру с ближайшим прототипом.

In [ ]:
new_cars = [
    ('Lada Granta',      90,   6.8,  1100),
    ('Kia K5',          194,   8.2,  1530),
    ('Lexus LX 600',    409,  13.0,  2600),
]
new_names = [c[0] for c in new_cars]
X_new = np.array([c[1:] for c in new_cars], dtype=float) / X_MAX

rows = []
for (nm, *vals), x in zip(new_cars, X_new):
    d_km = [dist(x, p) for p in km_prototypes]
    d_mm = [dist(x, p) for p in mm_prototypes]
    rows.append({'Объект': nm, 'Признаки': vals,
                 'K средних: расстояния': np.round(d_km, 3).tolist(),
                 'K средних: кластер': int(np.argmin(d_km)) + 1,
                 'Максимин: расстояния': np.round(d_mm, 3).tolist(),
                 'Максимин: кластер': int(np.argmin(d_mm)) + 1})
display(pd.DataFrame(rows).set_index('Объект'))

plot_clusters(X, km_labels, km_prototypes, 'Распознавание новых объектов (прототипы метода K средних)',
              new_objects=X_new, new_names=new_names)

### 6. Оценка вычислительной сложности

Обозначения: $N$ — количество объектов, $M$ — количество признаков, $K$ — количество кластеров, $I$ — количество итераций.

**Метод *K* средних.** На каждой итерации вычисляется $N \cdot K$ расстояний (по $O(M)$ операций каждое) и пересчитываются прототипы ($O(N \cdot M)$). Итого $O(I \cdot N \cdot K \cdot M)$, память $O(N \cdot M + K \cdot M)$.

**Метод максимина.** Поиск второго прототипа — $O(N \cdot M)$. Каждый шаг распределения требует $N \cdot K$ расстояний ($O(N K M)$) и пересчёта порога по всем парам прототипов ($O(K^2 M)$). Так как на каждом шаге добавляется хотя бы один кластер, число шагов не больше $K$, итого в худшем случае $O(N \cdot K^2 \cdot M)$; при $K \ll N$ на практике время близко к линейному по $N$.

Проверим оценки экспериментально: на случайных данных (3 «облака» точек) увеличиваем $N$ и измеряем время работы.

In [ ]:
def make_blobs(n, rng):
    centers = np.array([[0.2, 0.2, 0.2], [0.5, 0.8, 0.5], [0.85, 0.4, 0.9]])
    lab = rng.integers(0, 3, n)
    return centers[lab] + rng.normal(0, 0.06, (n, 3))

rng = np.random.default_rng(16)
sizes = [100, 200, 400, 800, 1600, 3200]
res = []
for n in sizes:
    data = make_blobs(n, rng)
    t0 = time.perf_counter(); _, _, it = kmeans(data, 3, verbose=False); t_km = time.perf_counter() - t0
    t0 = time.perf_counter(); _, pr, _, _ = maximin(data, verbose=False); t_mm = time.perf_counter() - t0
    res.append({'N': n, 'K средних, с': round(t_km, 4), 'итераций I': it,
                'с / (I·N)·1e6': round(t_km / (it * n) * 1e6, 2),
                'Максимин, с': round(t_mm, 4), 'K (максимин)': len(pr)})
df_t = pd.DataFrame(res)
display(df_t)

plt.figure(figsize=(7, 4))
plt.loglog(df_t['N'], df_t['K средних, с'], 'o-', label='K средних')
plt.loglog(df_t['N'], df_t['Максимин, с'], 's-', label='Максимин')
ref = df_t['K средних, с'].iloc[0] * np.array(sizes) / sizes[0]
plt.loglog(sizes, ref, 'k--', alpha=0.5, label='~ N (линейный рост)')
plt.xlabel('Количество объектов N')
plt.ylabel('Время, с')
plt.title('Зависимость времени работы от N (M = 3)')
plt.grid(alpha=0.3, which='both')
plt.legend()
plt.tight_layout()
plt.show()

### Вывод

В ходе выполнения лабораторной работы изучены методы и алгоритмы разделения объектов на кластеры. Разработаны программы кластеризации методом *K* средних (число кластеров задаётся заранее, прототипы — средние значения признаков) и методом максимина (число кластеров определяется автоматически по пороговому расстоянию, прототипы — объекты выборки). На выборке из 15 автомобилей метод *K* средних при $K = 3$ выделил группы «малолитражки», «седаны среднего класса» и «внедорожники»; метод максимина автоматически определил то же число кластеров ($K = 3$), но из-за того, что его прототипы — крайние объекты выборки, границы кластеров сместились; уточнение разбиения методом *K* средних от прототипов максимина дало итоговое разбиение. Полученные прототипы использованы для распознавания новых объектов. Сложность метода *K* средних — $O(I \cdot N \cdot K \cdot M)$, метода максимина — $O(N \cdot K^2 \cdot M)$; экспериментально время работы растёт примерно линейно по числу объектов.